# 09 — Target-Load Comparison and saturation

The E-Perf-1 Target-Load Delivery Summary is target-load evidence; E-Perf-10 is saturation evidence. Historical eKuiper QoS-0 tails are invalid comparator evidence and are not subtracted. N, units, `thesis_evidence=false`, and descriptive-only uncertainty are explicit; missing leaves remain PENDING, never zero.


In [ ]:
import json, os
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from wafer_analysis.focused import evidence_label, pending_record
from wafer_analysis.paths import resolve_result_batch

def passed_json(batch, artifact):
    rows = []
    for path in sorted(batch.rglob(artifact)):
        status = path.parent / 'canonical-status.json'
        if status.is_file() and json.loads(status.read_text()).get('status') == 'passed':
            rows.append((path, json.loads(path.read_text())))
    return rows

try: batch=resolve_result_batch('e-perf-10', diagnostic_path=os.environ.get('E_PERF_10_DIR'))
except (FileNotFoundError, RuntimeError, ValueError): batch=None
raw=[] if batch is None else [value for _,value in passed_json(batch,'rate-sweep.json')]
by_condition={(value['system'],value['offered_rate_msg_s']):value for value in raw}
rows=[]
for system in ['mqtt-loopback','native','wafer','ekuiper']:
    for rate in [500,1000,2000,4000,8000,16000]:
        value=by_condition.get((system,rate))
        if value is None:
            row=pending_record(f'{system} at {rate} messages/second','no passed rate-sweep.json leaf','messages/second and milliseconds')
            row.update({'system':system,'offered_rate_msg_s':rate}); rows.append(row)
        else:
            rows.append({'question':f'{system} at {rate} messages/second','status':'READY','system':system,'offered_rate_msg_s':rate,'achieved_rate_msg_s':value['achieved_rate_msg_s'],'p95_ms':value['latency_ns']['p95']/1e6,'p99_ms':value['latency_ns']['p99']/1e6,'loss_percent':value['loss_percent'],'units':'messages/second, milliseconds, percent','uncertainty':'descriptive only','thesis_evidence':False})
df=pd.DataFrame(rows); print(evidence_label(len(df[df.status=='READY']), 'messages/second, milliseconds, percent', False)); display(df)
ready=df[df.status=='READY']
if not ready.empty:
    for system, group in ready.groupby('system'):
        plt.plot(group['offered_rate_msg_s'],group['p99_ms'],marker='o',label=system)
    plt.xlabel('Offered rate (messages/second)'); plt.ylabel('p99 latency (ms)'); plt.legend(); plt.title('Focused saturation diagnostic')
